# 03. LOT별 온도와 출력 비교

`TC-Z3`, `OP-Z3`, `TC-Z4`, `OP-Z4`, `TC-Z5`, `OP-Z5`, `CP-Z4`를 비교함. `VIEW_LOT`로 화면에 볼 LOT를 고르며 14개 LOT의 그림은 같은 파일명으로 저장함.

- 가로축은 실제 측정 날짜와 시각임. `01/02 13:26`은 1월 2일 오후 1시 26분임.
- 진한 선은 해당 분의 평균, 옅은 색은 최소~최대임. 관측 없는 분은 비워 두고 원자료의 출력 0은 유지함.
- 온도 ℃, 출력 %는 사전의 가정임. `CP-Z4`는 단위가 미확인이라 기록값으로 표시함.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display

# 저장소 루트, notebooks 폴더, 개인 노트북 폴더에서 같은 파일을 읽음.
if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')

TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(DATA_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(DATA_DIR, 'T-CR1-WLD01_용접.csv')  # 현재 분석에서는 용접 파일을 읽지 않음.
ENC = 'utf-8-sig'

FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark', '03_lot_comparison')

# CSV를 직접 읽고, 계산에 사용하는 날짜 컬럼을 날짜형으로 바꿈.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 읽은 원본 데이터프레임을 이름별로 보관함.
sources = {
    'temp': temp,
    'event': event,
    'repair': repair,
    'eqp': eqp,
    'part': part,
    'maint': maint,
    'partchg': partchg,
}

# 설치된 한글 폰트를 사용함.
font_names = []

for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)

for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if font_name in font_names:
        rcParams['font.family'] = font_name
        break
else:
    print('경고: 사용할 수 있는 한글 폰트를 찾지 못했음. 그래프의 한글이 깨질 수 있음.')

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False

TARGETS = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "TC-Z5", "OP-Z5", "CP-Z4"]

raw = sources["temp"].sort_values(["LOT_NO", "MEAS_DT"])


In [ ]:
# LOT 관측 범위와 강종을 연결함.
# LOT의 실제 첫 관측과 마지막 관측을 구함.

lot_bounds = raw.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    ROWS=("MEAS_DT", "size"),
)

# CR1 강종변경 중 LOT 첫 관측과 시각이 정확히 같은 기록만 연결함.
lot_grade_changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_bounds = lot_bounds.merge(
    lot_grade_changes[["EVT_DT", "AFT_VAL"]],
    left_on="START", right_on="EVT_DT", how="left", validate="many_to_one",
)

lot_bounds = lot_bounds.rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")



In [ ]:
# LOT별 관측 기간과 강종을 정리함.

lot_bounds["DURATION_H"] = (lot_bounds["END"] - lot_bounds["START"]).dt.total_seconds() / 3600

lot_context = lot_bounds[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]

lot_context = lot_context.sort_values("START").reset_index(drop=True)

lots = lot_context.set_index("LOT_NO")[["START", "END", "GRADE"]]
lots = lots.rename(columns={"START": "시작", "END": "끝", "GRADE": "강종"})


In [ ]:
# LOT별 분 단위 통계를 계산함.

# LOT마다 따로 집계하므로 서로 다른 LOT 사이의 빈 날짜를 메우지 않음
# OP의 0은 실제 출력임. 결측으로 바꾸지 않음
means = {}
lows = {}
highs = {}
counts = {}

for lot, rows in raw.groupby("LOT_NO"):
    minutes = rows.set_index("MEAS_DT")[TARGETS].resample("1min")

    means[lot] = minutes.mean()
    lows[lot] = minutes.min()
    highs[lot] = minutes.max()
    counts[lot] = minutes.count()

# 아래 2개는 팀에서 확정한 해석임. 전체 LOT를 고장으로 표시하는 뜻은 아님
operation = {240058: "정기수리를 위한 정지", 240061: "정기수리 후 재가동"}


## 1. LOT의 강종과 관측 수

일부만 기록된 분도 평균은 남기고 관측 수를 함께 확인함.


In [ ]:
display(lots)

quality = []

for lot, count in counts.items():
    quality.append({
        "LOT": lot,
        "강종": lots.loc[lot, "강종"],
        "전체 분 수": len(count),
        "센서별 관측 수의 최솟값": count.min().min(),
        "센서별 관측 수의 최댓값": count.max().max(),
        "관측이 하나도 없는 센서·분 수": count.eq(0).sum().sum(),
    })

display(pd.DataFrame(quality))

## 2. 온도끼리, 출력끼리, 같은 구역끼리 비교

같은 LOT의 Z3, Z4, Z5 온도축 범위는 같음. 온도와 출력을 겹친 그림은 양쪽 축이 다르므로 선의 교차 위치가 아닌 변화 시점을 비교함.


In [ ]:
# 화면에는 이 LOT만 표시함. 다른 LOT도 파일로는 모두 저장함
VIEW_LOT = 240001
ZONE_COLORS = {3: "#2B70B2", 4: "#D17A25", 5: "#448A65"}

# 같은 LOT의 세 구역을 같은 온도 범위에서 보기 위함
for lot, mean in means.items():
    low, high = lows[lot], highs[lot]
    time = mean.index

    grade = lots.loc[lot, "강종"]

    note = operation.get(lot, "")
    title = ('LOT ' + str(lot) + ', ' + str(grade))

    if note:
        title += (', ' + str(note))

    tc_min = low[["TC-Z3", "TC-Z4", "TC-Z5"]].min().min() - 1
    tc_max = high[["TC-Z3", "TC-Z4", "TC-Z5"]].max().max() + 1

    # 온도끼리, 출력끼리 겹쳐서 구역 차이를 보기 위함
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")

    for ax, prefix, unit in zip(axes, ["TC", "OP"], ["온도 (℃)", "출력 (%)"]):
        for zone, color in ZONE_COLORS.items():
            sensor = (str(prefix) + '-Z' + str(zone))
            ax.plot(time, mean[sensor], color=color, linewidth=1, label=sensor)
            ax.fill_between(time, low[sensor], high[sensor], color=color, alpha=0.10)

        ax.set_ylabel(unit)
        ax.legend(loc="upper left", ncol=3)
        ax.grid(alpha=0.15)

    axes[0].set_ylim(tc_min, tc_max)
    axes[1].set_ylim(0, 105)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[1].set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
    fig.suptitle(title + " | Z3, Z4, Z5 비교")
    output = os.path.join(FIG_DIR, '구역_간_비교', 'LOT별')

    os.makedirs(output, exist_ok=True)

    fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_온도_출력_비교.png')), dpi=140)

    if lot == VIEW_LOT:
        plt.show()

    plt.close(fig)

    # 온도와 출력은 단위가 다름. 왼쪽 온도축과 오른쪽 출력축으로 구분함
    for zone in [3, 4, 5]:
        tc, op = ('TC-Z' + str(zone)), ('OP-Z' + str(zone))
        fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
        right = ax.twinx()
        ax.plot(time, mean[tc], color="#2B70B2", linewidth=1, label=tc)
        ax.fill_between(time, low[tc], high[tc], color="#2B70B2", alpha=0.13)
        right.plot(time, mean[op], color="#D17A25", linewidth=1, label=op)
        right.fill_between(time, low[op], high[op], color="#D17A25", alpha=0.10)
        ax.set_ylim(tc_min, tc_max)
        right.set_ylim(0, 105)
        ax.set_ylabel("온도 (℃)", color="#2B70B2")
        right.set_ylabel("출력 (%)", color="#D17A25")
        ax.legend(loc="upper left")
        right.legend(loc="upper right")
        ax.grid(alpha=0.15)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        ax.set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
        ax.set_title(title + (' | Z' + str(zone) + ' 온도, 출력'))
        output = os.path.join(FIG_DIR, ('Z' + str(zone) + '_온도_출력'), 'LOT별')

        os.makedirs(output, exist_ok=True)

        fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_Z' + str(zone) + '_온도_출력.png')), dpi=140)

        if lot == VIEW_LOT:
            plt.show()

        plt.close(fig)

    # CP는 온도, 출력과 단위가 달라 별도 그림으로 같은 시각을 확인함
    fig, ax = plt.subplots(figsize=(13, 3.5), layout="constrained")
    ax.plot(time, mean["CP-Z4"], color="#7559A6", linewidth=1, label="CP-Z4")
    ax.fill_between(time, low["CP-Z4"], high["CP-Z4"], color="#7559A6", alpha=0.15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    ax.set_ylabel("CP-Z4 기록값 (단위 미확인)")
    ax.set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
    ax.set_title(title + " | CP-Z4")
    ax.grid(alpha=0.15)
    ax.legend()
    output = os.path.join(FIG_DIR, 'CP_기록값', 'LOT별')

    os.makedirs(output, exist_ok=True)

    fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_CP_Z4.png')), dpi=140)

    if lot == VIEW_LOT:
        plt.show()

    plt.close(fig)

## 3. LOT별 확인한 내용

| LOT, 강종 | 관찰과 해석 |
| --- | --- |
| 240001, SGCC | Z3, Z4 온도가 오를 때 출력이 낮아졌음. 제어 반응으로 볼 수 있으나 원인은 미확정이며, SGCC 1개 LOT만으로 정상 범위를 정하지 않음. |
| 240024, SPFH590 | `CP-Z4`의 짧은 하강과 가까운 시각 Z5 온도 상승, 출력 감소가 보였음. 특정 고장은 미확정임. |
| 240037, SPCE | `CP-Z4`가 올랐다 약 0.45로 돌아왔고, 가까운 시각 Z4, Z5 온도는 오르고 출력은 줄었음. 동시 변화만으로 CP를 원인으로 정하지 않음. |
| 240038, SPCEN | `TC-Z4`는 114와 비슷해도 `OP-Z4`는 114 후반 하강 전보다 높고 CP 중앙값은 비슷했음. 짧은 CP 상승도 있었지만 SPCEN의 필수 출력이나 고장을 확정하지 않음. |
| 240044, SPHC | 후반 Z3 온도와 출력이 반대로 움직였고 출력은 약 99.37%에서 평평하게 이어졌음. 실제 설정 상한은 미확인이며 이 값만으로 LOT 전체를 제외하지 않음. |
| 240058, DP590 | **정기수리 때문에 소둔로를 멈춘 LOT이며 후반 온도 하강은 정지 과정임.** CP는 온도보다 먼저 낮아졌고, 뒤의 오일씰과 O링 교체만으로 CP 하강 원인을 확정하지 않음. |
| 240061, SPHC | **정기수리 후 첫 재가동 LOT이며 초기 온도 상승은 재가동 과정임.** 온도 상승 후에도 높은 `OP-Z4`, 이후 `OP-Z3` 0과 온도 왕복은 별도로 검토함. |
| 240069, SPHC | CP 최대 상승 때 Z4 온도는 오르고 출력은 낮아졌으며, 이후 Z3 출력 0과 온도 왕복도 있었음. 나중의 절연 정비와 O링 교체를 CP 고장이나 누설 증거로 단정하지 않음. |
| 240091, SPHC | CP의 짧은 이탈과 후반 Z3 출력 0, 온도 왕복을 따로 검토함. 출력 0과 높은 평탄값만으로 고장이나 설정 상한을 확정하지 않음. |
| 240108, SPFH590 | 11:27 Z3 출력 저하 뒤 11:29 온도가 최고였고, Z4 출력은 같은 강종인 024와 비교함. 제어 반응이나 지연으로 볼 수 있으나 소재 이동시간과 원인은 미확정임. |
| 240114, SPCEN | 하강 전 Z4 온도는 038과 비슷하지만 출력은 낮았고, 후반 온도와 출력 변화에도 CP는 약 0.45였음. CP도 낮아진 058과 같은 사건으로 묶지 않음. |
| 240117, SPCD | 초기 온도 상승 뒤 정비가 겹쳤고 오후에는 CP가 따로 낮아졌음. 정비가 원인인지는 미확정이며 SPCD는 이 LOT 하나뿐임. |
| 240121, SPCE | `TC-Z3` 최고는 16:42, CP 중앙값은 다른 SPCE와 비슷했음. 냉각과 과시효 채널의 변화를 이번 7개 채널의 이상 근거로 옮기지 않음. |
| 240133, SPCE | 15:54 정비 감지 전후 CP 중앙값은 비슷했고 정비는 온도 관측 종료 뒤 16:54에 시작했음. 감지 이후 전체를 이상으로 보거나 수리 효과를 추정하지 않음. |

LOT 간 비교는 관측 후의 해석이며, 다른 시점의 정상 범위나 고장 정답을 확정한 결과는 아님.


구간별 판단은 [05 구간 검토](05_interval_review.ipynb)에 모음. 제외 표시가 없다고 정상으로 확정한 것은 아님.


### 실제 시각과 경과시간

240001은 2024-01-02 03:36에 관측을 시작했으므로 `13:26`은 오후 1시 26분, 첫 관측 후 590분임. 표의 `OP-Z3 4.30% / TC-Z3 866.84℃`는 13:26 이상 13:27 미만의 평균이며, 13:29 최고 온도와 다름.

그림은 분 시작에 평균을 표시하지만 계산에는 해당 분 전체가 필요함. 원자료 한 점, 1분 평균, 1분 중앙값을 구분함.

수리 중 강종변경은 6/11 DP590→SPCC, 6/13 SPCC→DP590이며 6/15의 061 시작에 DP590→SPHC로 바뀜. 수리 종료 후 061 전에는 중간 강종변경이 없으므로 058 정지, 061 첫 재가동으로 판단하며, 관측 없는 059, 060을 추정해 채우지 않음.


### 다른 자료와 겹친 시간

온도 LOT별 첫~마지막 관측 구간 합은 약 4.12일, 용접 기록이 있는 날짜는 77일이며 실제 조업일수는 아님. 온도와 겹친 용접 접합번호는 82개(`240108`, `240114`, `240117`, `240121`)이나 같은 소재인지는 확인되지 않음.

설비와 강종 연결은 [01](01_data_alignment.ipynb), 정비와 부품 연결은 [02](02_maintenance_parts.ipynb)에서 확인함.
